# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) Mass Spectrometry dataset using the `mlcroissant` library, based on the Croissant data packaging standard.

### Dataset Source
The dataset is described by a Croissant schema accessible via the URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')  # Suppress pandas warnings for cleaner output

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets and their fields using their `@id` values. This helps locate the correct identifiers for programmatic access.

In [ ]:
# List all record sets by @id
record_sets = list(meta.record_sets)
print('Record Sets:')
for rs in record_sets:
    print(f"  - Name: {rs.name} | @id: {rs.id}")

# Show fields for each record set (by their @id)
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (id: {rs.id}) - Fields:")
    for field in rs.fields:
        print(f"    - {field.name:30s} | @id: {field.id:40s} | dtype: {field.data_type}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame, referencing record sets and fields by their `@id`.

> **Tip:** Make sure you use the correct `@id` values for record sets/fields. Use outputs from the previous cell to identify them.

In [ ]:
# Create a dictionary of DataFrames for each record set
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    # Load records using mlcroissant
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set '@id': {rs_id}; shape: {df.shape}")
    else:
        print(f"[Warning] No records found for record set '@id': {rs_id}")

# For demonstration, show columns from the first available record set
example_rs_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if example_rs_id and example_rs_id in dataframes:
    print(f"\nExample DataFrame columns for record set {example_rs_id}:")
    print(dataframes[example_rs_id].columns.tolist())
    dataframes[example_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process one of the main record sets that contains protein measurements. We'll filter and normalize a numeric field, and group by another attribute, referencing all by `@id`.

For demonstration, let's assume the most important record set is the first one listed above and:
- `Numeric Field`: Choose a numeric field (like coverage, peptide_count, molecular_weight, etc.) by its `@id` from the outputs above.
- `Group Field`: Choose a field to group by (such as sample, accession, protein_id, etc.), by its `@id`.

In [ ]:
# Pick the first available record set for demonstration
record_set_to_analyze = example_rs_id
if record_set_to_analyze and record_set_to_analyze in dataframes:
    df = dataframes[record_set_to_analyze].copy()
    print(f"Analyzing record set: {record_set_to_analyze}")

    print(f"Available columns (field @ids): {df.columns.tolist()}")
    # You may need to adjust these @ids. For our demo, let's guess likely ones:
    # We'll pick the first numeric-looking column available
    import numpy as np

    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            print(f"Using numeric field: {numeric_field_id}")
            break

    if not numeric_field_id:
        print("No numeric field detected for EDA!")
    else:
        # Use a basic threshold to filter
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].nunique() > 10 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized field '{numeric_field_id}':")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a string/categorical field for grouping
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                print(f"Using group field: {group_field_id}")
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df.head())
else:
    print("No data frame available for analysis.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalization, as well as possible group-wise summaries.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field_id:
    fig, ax = plt.subplots(1,2, figsize=(12,4))
    sns.histplot(filtered_df[numeric_field_id], ax=ax[0], kde=True, color='skyblue')
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], ax=ax[1], kde=True, color='orange')
    ax[1].set_title(f"Normalized {numeric_field_id}")
    plt.tight_layout()
    plt.show()

    # Optional: group bar plot if grouping was performed
    if 'grouped_df' in locals() and group_field_id:
        grouped_df.head(10).plot(kind='bar', title=f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()
else:
    print("No filtered dataframe or numeric field for visualization.")

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to:
- Access and introspect the FAIR^2 dataset's structure using Croissant schema `@id` references
- Extract data from record sets and load them into DataFrames
- Perform simple filtering and normalization operations on numeric fields
- Visualize data distributions and grouped means

> For your own analysis, iterate through the record sets and fields relevant to your biological or data science question. Refer to the `@id` values to ensure consistent, schema-compliant access to all dataset elements.